## Imports y utilidades


In [1]:
import pandas as pd
import numpy as np
import yaml
from sqlalchemy import create_engine, text
import hashlib
from datetime import datetime

## Conexiones a las bases de datos


In [2]:
with open('../config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config.get('CO_SA')
    config_etl = config.get('ETL_PRO')

url_co = (f"{config_co['drivername']}://{config_co['user']}@{config_co['host']}:{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}@{config_etl['host']}:{config_etl['port']}/{config_etl['dbname']}")

co_sa = create_engine((f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:{config_co['port']}/{config_co['dbname']}"))
etl_conn = create_engine((f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:{config_etl['port']}/{config_etl['dbname']}"))

print('Conexiones creadas. Revise `config.yml` si hay errores en las credenciales.')

Conexiones creadas. Revise `config.yml` si hay errores en las credenciales.


## Explorar la tabla fuente `mensajeria_novedadesservicio`

In [3]:
# Cargar una muestra de la tabla de novedades (origen)
try:
    df_novedad = pd.read_sql_table('mensajeria_novedadesservicio', con=co_sa)
    print('Tabla cargada desde origen: mensajeria_novedadesservicio')
    display(df_novedad.head())
    print('Información de columnas:')
    print(df_novedad.info())
except Exception as e:
    print('Error al leer mensajeria_novedadesservicio:', e)
    df_novedad = pd.DataFrame()

Tabla cargada desde origen: mensajeria_novedadesservicio


,id,fecha_novedad,tipo_novedad_id,descripcion,servicio_id,es_prueba,mensajero_id
0,4,2023-11-30 05:00:00+00:00,1,A,51,True,7
1,5,2023-11-30 05:00:00+00:00,1,Halo,51,True,7
2,6,2023-11-30 05:00:00+00:00,1,A,51,True,7
3,7,2023-11-30 05:00:00+00:00,1,B,51,True,7
4,8,2023-11-30 05:00:00+00:00,1,A,51,True,7


Información de columnas:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5208 entries, 0 to 5207
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   id               5208 non-null   int64              
 1   fecha_novedad    5208 non-null   datetime64[ns, UTC]
 2   tipo_novedad_id  5208 non-null   int64              
 3   descripcion      5208 non-null   object             
 4   servicio_id      5208 non-null   int64              
 5   es_prueba        5208 non-null   bool               
 6   mensajero_id     5208 non-null   int64              
dtypes: bool(1), datetime64[ns, UTC](1), int64(4), object(1)
memory usage: 249.3+ KB
None


## Leer dimensiones necesarias desde la bodega ETL


In [ ]:
# Cargar dimensiones (si existen)

def safe_read_table(name):
    try:
        return pd.read_sql_table(name, con=etl_conn)
    except Exception as e:
        print(f'No se pudo leer {name}:', e)
        return pd.DataFrame()

dim_fecha = safe_read_table('dim_fecha')
dim_hora = safe_read_table('dim_hora')
dim_cliente = safe_read_table('dim_cliente')
dim_novedad = safe_read_table('dim_novedad')
dim_servicio = safe_read_table('dim_servicio')
dim_mensajero = safe_read_table('dim_mensajero')

print("Dimensiones leídas")

print("dim_fecha", dim_fecha.shape)
print("dim_hora", dim_hora.shape)
print("dim_cliente", dim_cliente.shape)
print("dim_novedad", dim_novedad.shape)
print("dim_servicio", dim_servicio.shape)
print("dim_mensajero", dim_mensajero.shape)

# ==========================================================
# CONSTRUCCIÓN DE LA TABLA DE HECHOS NOVEDAD
# ==========================================================

print("Construyendo tabla de hechos...")

# -----------------------------
# Copia de datos del origen
# -----------------------------
df_hecho = df_novedad.copy()

# -----------------------------------
# Obtener cliente desde el servicio
# -----------------------------------

servicio = pd.read_sql_table(
    "mensajeria_servicio",
    con=co_sa
)

servicio_aux = servicio[
    [
        "id",
        "cliente_id"
    ]
].rename(columns={
    "id": "servicio_id"
})

df_hecho = df_hecho.merge(
    servicio_aux,
    on="servicio_id",
    how="left"
)

# Asegurar columnas necesarias aunque la fuente no las tenga
required_cols = ["id", "fecha_novedad", "tipo_novedad_id", "mensajero_id"]
for col in required_cols:
    if col not in df_hecho.columns:
        raise KeyError(f"Falta la columna requerida '{col}' en la tabla fuente")

# Normalizar el identificador del cliente para evitar conflictos de tipos en el merge
df_hecho["cliente_id"] = pd.to_numeric(df_hecho["cliente_id"], errors="coerce").astype("Int64")

# Seleccionar únicamente las columnas necesarias
df_hecho = df_hecho[
[
"id",
"fecha_novedad",
"tipo_novedad_id",
"descripcion",
"mensajero_id",
"cliente_id"
]
].copy()

# Cada registro representa una novedad
df_hecho["numero_novedades"] = 1

# -----------------------------
# Preparación de fechas
# -----------------------------

df_hecho["fecha_novedad"] = pd.to_datetime(
    df_hecho["fecha_novedad"],
    utc=True,
    errors="coerce"
).dt.tz_localize(None)

df_hecho["fecha"] = df_hecho["fecha_novedad"].dt.normalize()

df_hecho["hora"] = df_hecho["fecha_novedad"].dt.hour
df_hecho["minuto"] = df_hecho["fecha_novedad"].dt.minute

# Normalizar dimensión fecha

if not dim_fecha.empty and "fecha_completa" in dim_fecha.columns:
    dim_fecha["fecha_completa"] = pd.to_datetime(
        dim_fecha["fecha_completa"],
        errors="coerce"
    ).dt.normalize()

# -----------------------------
# Merge Dim Fecha
# -----------------------------

df_hecho = df_hecho.merge(

    dim_fecha[
        [
            "key_dim_fecha",
            "fecha_completa"
        ]
    ] if not dim_fecha.empty and "fecha_completa" in dim_fecha.columns else pd.DataFrame(columns=["key_dim_fecha", "fecha_completa"]),

    left_on="fecha",

    right_on="fecha_completa",

    how="left"
)

# -----------------------------
# Merge Dim Hora
# -----------------------------

df_hecho = df_hecho.merge(

    dim_hora[
        [
            "key_dim_hora",
            "hora",
            "minuto"
        ]
    ] if not dim_hora.empty and {"key_dim_hora", "hora", "minuto"}.issubset(dim_hora.columns) else pd.DataFrame(columns=["key_dim_hora", "hora", "minuto"]),

    on=[
        "hora",
        "minuto"
    ],

    how="left"
)

# Merge Dim Cliente
dim_cliente_aux = (
    dim_cliente[["key_dim_cliente", "id_cliente"]].copy()
    if not dim_cliente.empty and {"key_dim_cliente", "id_cliente"}.issubset(dim_cliente.columns)
    else pd.DataFrame(columns=["key_dim_cliente", "id_cliente"])
)

if not dim_cliente_aux.empty:
    dim_cliente_aux["id_cliente"] = pd.to_numeric(dim_cliente_aux["id_cliente"], errors="coerce").astype("Int64")

df_hecho = df_hecho.merge(
    dim_cliente_aux,
    left_on="cliente_id",
    right_on="id_cliente",
    how="left"
)

# =====================================================
# Merge con dimensión novedad
# =====================================================

dim_novedad_aux = (
    dim_novedad[
        [
            "key_dim_novedad",
            "id_novedad",
            "descripcion_novedad"
        ]
    ]
)

df_hecho = df_hecho.merge(
    dim_novedad_aux,
    left_on=[
        "tipo_novedad_id",
        "descripcion"
    ],
    right_on=[
        "id_novedad",
        "descripcion_novedad"
    ],
    how="left"
)

# -----------------------------
# Merge Dim Mensajero
# -----------------------------

if not dim_mensajero.empty and {"key_dim_mensajero", "id_mensajero"}.issubset(dim_mensajero.columns):
    dim_mensajero_aux = (
        dim_mensajero
        .sort_values("key_dim_mensajero")
        .drop_duplicates("id_mensajero")
    )
else:
    dim_mensajero_aux = pd.DataFrame(columns=["key_dim_mensajero", "id_mensajero"])

df_hecho = df_hecho.merge(

    dim_mensajero_aux[
        [
            "key_dim_mensajero",
            "id_mensajero"
        ]
    ],

    left_on="mensajero_id",

    right_on="id_mensajero",

    how="left"
)

# -----------------------------
# Construcción del hecho final
# -----------------------------

hechos_novedad = df_hecho[
    [
        "key_dim_fecha",
        "key_dim_hora",
        "key_dim_cliente",
        "key_dim_novedad",
        "key_dim_mensajero",
        "numero_novedades"
    ]
].copy()

# -----------------------------
# Validaciones
# -----------------------------

print("\n==============================")
print("VALIDACIONES")
print("==============================")

print("Registros:", len(hechos_novedad))

print("\nValores nulos")

print(
    hechos_novedad[
        [
            "key_dim_fecha",
            "key_dim_hora",
            "key_dim_cliente",
            "key_dim_novedad",
            "key_dim_mensajero"
        ]
    ].isnull().sum()
)

print("\nDuplicados")

print(hechos_novedad.duplicated().sum())

print("\nVista previa")

display(hechos_novedad.head())

Dimensiones leídas
dim_fecha (1461, 15)
dim_hora (1440, 7)
dim_cliente (27, 8)
dim_novedad (3931, 4)
dim_servicio (28430, 2)
dim_mensajero (50, 6)
Construyendo tabla de hechos...


In [ ]:
sql = """
DROP TABLE IF EXISTS hechos_novedad;

CREATE TABLE hechos_novedad(

    key_dim_fecha INTEGER,

    key_dim_hora INTEGER,

    key_dim_cliente INTEGER,

    key_dim_novedad INTEGER,

    key_dim_mensajero INTEGER,

    numero_novedades INTEGER

);
"""

with etl_conn.begin() as conn:
    conn.execute(text(sql))

print("Tabla creada.")

In [ ]:
hechos_novedad.to_sql(
    "hechos_novedad",
    con=etl_conn,
    if_exists="append",
    index=False
)

print("Tabla hechos_novedad cargada correctamente.")